In [1]:
import kagglehub
from confluent_kafka.admin import AdminClient, NewTopic
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

spark = (SparkSession.builder.appName("MyApp")
    .config("spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3")    
    .master("local[*]")
    .getOrCreate())

/home/duyhuynh/Desktop/school/bigdata/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
26/04/19 15:10:47 WARN Utils: Your hostname, debian resolves to a loopback address: 127.0.1.1; using 192.168.122.1 instead (on interface virbr0)
26/04/19 15:10:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/duyhuynh/Desktop/school/bigdata/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/duyhuynh/.ivy2/cache
The jars for the packages stored in: /home/duyhuynh/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a46b7770-9ee9-411c-bb6d-15c7e335574e;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.3 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in local-m2-cache
	found org.xerial.snappy#snappy-java;1.1.10.5 in local-m2-cache
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in local-m2-cache
	found com.google.code.findbugs#jsr305;3.0.0 in local-m2-cache
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolv

In [2]:
path = kagglehub.dataset_download("grouplens/movielens-latest-small")
df_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_tags = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)
df_ratings.show(1)
df_movies.show(1)
df_tags.show(1)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
+------+-------+------+---------+
only showing top 1 row

+-------+----------------+--------------------+
|movieId|           title|              genres|
+-------+----------------+--------------------+
|      1|Toy Story (1995)|Adventure|Animati...|
+-------+----------------+--------------------+
only showing top 1 row

+------+-------+-----+----------+
|userId|movieId|  tag| timestamp|
+------+-------+-----+----------+
|     2|  60756|funny|1445714994|
+------+-------+-----+----------+
only showing top 1 row



In [3]:
bootstrap_servers = "localhost:30090,localhost:30091,localhost:30092"
# First, delete topics if they exist
topics = [ 'Lab1_' + t for t in['ratings', 'movies', 'tags']]

admin_client = AdminClient({'bootstrap.servers': bootstrap_servers})
admin_client.delete_topics(topics, operation_timeout=10)
# Create new topics
new_topics = [NewTopic(topic=t, num_partitions=1, replication_factor=3) for t in topics]
fs = admin_client.create_topics(new_topics)
for topic, f in fs.items():
    try:
        f.result()
        print(f"Topic {topic} created")
    except Exception as e:
        print(f"Failed to create topic {topic}: {e}")

Topic Lab1_ratings created
Topic Lab1_movies created
Topic Lab1_tags created


In [4]:
df_ratings.selectExpr("to_json(struct(*)) AS value") \
.write \
.format("kafka") \
.option("kafka.bootstrap.servers", bootstrap_servers) \
.option("topic", topics[0]) \
.save()

df_movies.selectExpr("to_json(struct(*)) AS value") \
.write \
.format("kafka") \
.option("kafka.bootstrap.servers", bootstrap_servers) \
.option("topic", topics[1]) \
.save()

df_tags.selectExpr("to_json(struct(*)) AS value") \
.write \
.format("kafka") \
.option("kafka.bootstrap.servers", bootstrap_servers) \
.option("topic", topics[2]) \
.save()

In [5]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, LongType
from pyspark.sql.functions import col, from_json, to_timestamp, explode, split, window, count, dense_rank, desc, expr
from pyspark.sql.window import Window

movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", LongType(), True) 
])

tags_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp", LongType(), True)
])

df_movies = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", bootstrap_servers) \
    .option("subscribe", "Lab1_movies") \
    .option("startingOffsets", "earliest") \
    .load() \
    .selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), movies_schema).alias("data")) \
    .select("data.*")

df_ratings_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", bootstrap_servers) \
    .option("subscribe", "Lab1_ratings") \
    .option("startingOffsets", "earliest") \
    .option("maxOffsetsPerTrigger", 100) \
    .load() \
    .selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), ratings_schema).alias("data")) \
    .select("data.*") \
    .withColumn("ratingTime", to_timestamp(col("timestamp")))

df_tags_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", bootstrap_servers) \
    .option("subscribe", "Lab1_tags") \
    .option("startingOffsets", "earliest") \
    .option("maxOffsetsPerTrigger", 100) \
    .load() \
    .selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), tags_schema).alias("data")) \
    .select("data.*") \
    .withColumn("tagTime", to_timestamp(col("timestamp")))

In [6]:
df_movies_exploded = df_movies.withColumn("genre", explode(split(col("genres"), "\\|")))

hot_genres_df = df_ratings_stream.join(df_movies_exploded, "movieId") \
    .groupBy("genre") \
    .count()

query_ex2 = hot_genres_df.writeStream \
    .outputMode("complete") \
    .format("console") \
    .trigger(processingTime="5 seconds") \
    .start()

26/04/19 15:10:59 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-af77506e-f792-430e-88d5-0e944f86265c. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/19 15:10:59 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [ ]:
df_ratings_watermarked = df_ratings_stream.withWatermark("ratingTime", "10 minutes")

windowed_counts = df_ratings_watermarked \
    .groupBy(window(col("ratingTime"), "5 minutes"), col("movieId")) \
    .count() \
    .withColumnRenamed("count", "cnt")

joined_trends = windowed_counts.join(df_movies, "movieId")

window_spec = Window.partitionBy("window").orderBy(desc("cnt"), col("movieId").asc())

trending_now_df = joined_trends.withColumn("rank", dense_rank().over(window_spec)) \
    .filter(col("rank") <= 3) \
    .select("window", "movieId", "title", "cnt", "rank")

query_ex3 = trending_now_df.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime="5 seconds") \
    .start()

26/04/19 15:11:00 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-421889d2-ccf1-47f2-93b5-8924c86ef800. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/19 15:11:00 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


AnalysisException: [NON_TIME_WINDOW_NOT_SUPPORTED_IN_STREAMING] Window function is not supported in DENSE_RANK(CNT#299L, MOVIEID#207) (as column `rank`) on streaming DataFrames/Datasets. Structured Streaming only supports time-window aggregation using the WINDOW function. (window specification: (PARTITION BY WINDOW ORDER BY CNT DESC NULLS LAST, MOVIEID ASC NULLS FIRST ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW))

In [ ]:
ratings_wm = df_ratings_stream.withWatermark("ratingTime", "1 hours")
tags_wm = df_tags_stream.withWatermark("tagTime", "1 hours")

stream_join_df = ratings_wm.join(
    tags_wm,
    expr("""
        ratings_wm.movieId = tags_wm.movieId AND
        tagTime >= ratingTime AND
        tagTime <= ratingTime + interval 1 hours
    """)
).select(
    ratings_wm["movieId"],
    tags_wm["tag"],
    ratings_wm["rating"],
    ratings_wm["ratingTime"],
    tags_wm["tagTime"]
)

query_ex4 = stream_join_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .trigger(processingTime="5 seconds") \
    .start()